# mT5-base — english + draft → competition register (T4 run)

Tests whether feeding the **English source** (the common ancestor of both translations) beats the
draft-only incumbent.

| | Token F1 | ROUGE-L | LB |
|---|---|---|---|
| **incumbent** — BanglaT5, draft only | **0.7724** | **0.7324** | **0.85030** |
| draft + 3 regexes, no model | 0.5984 | 0.5482 | 0.7564 |
| **this run** | ? | | |

## ⚠️ This is a COMPROMISED configuration — read before trusting the result

A T4 cannot run mT5-base at the lengths this data needs. Measured mT5 token lengths:
source p99 **992**, target p99 **575**. The proper config is `1024/640`; this run uses **640/448**.

| cap | source truncated | target truncated |
|---|---|---|
| **640/448 (this run)** | **~9%** | **~1%** |
| 1024/640 (proper) | 0.7% | 0.1% |

**~9% of inputs lose their tail.** Truncated source is information the model can never recover, so a
weak result here is partly a measure of the truncation, not only of the hypothesis.

**Also note mT5 is not the best carrier for this task.** Measured, both real tokenizers:

| | source tokens | target tokens |
|---|---|---|
| BanglaT5 | **364** | **145** |
| mT5 | 443 | **272** |

mT5 needs **88% more tokens for the same Bengali output**, where generation quality lives. The
higher-priority experiment is **BanglaT5 + english+draft**, which changes one variable against a
model with a measured LB score. See `fine_tune_MT5_project/` for the full run order.

**Treat this run as an early signal, not the verdict.**

In [ ]:
# ══ 1 — hardware gate ═══════════════════════════════════════════════════════
import torch
n = torch.cuda.device_count()
assert n > 0, "No GPU. Settings -> Accelerator -> GPU T4"
cap = torch.cuda.get_device_capability(0)
print(f"torch {torch.__version__} | {[torch.cuda.get_device_name(i) for i in range(n)]} | sm_{cap[0]}{cap[1]}")
assert cap[0] >= 7, f"WRONG ACCELERATOR sm_{cap[0]}{cap[1]} — Kaggle's PyTorch has no sm_60 kernels."
print(f"bf16 hardware: {cap[0] >= 8}  ->  precision will be {'bf16' if cap[0] >= 8 else 'fp32'}")
print("✅ hardware OK")

In [ ]:
# ══ 2 — pinned libs (trap #00) ══════════════════════════════════════════════
!pip install -q --upgrade "transformers==4.57.3" git+https://github.com/csebuetnlp/normalizer
import transformers
assert transformers.__version__ == "4.57.3", transformers.__version__
print("transformers", transformers.__version__)

In [ ]:
# ══ 3 — locate code + prepared data ═════════════════════════════════════════
import glob, os, shutil, sys, subprocess
print("/kaggle/input:", os.listdir("/kaggle/input"))

hits = glob.glob("/kaggle/input/**/02_train_t5.py", recursive=True)
assert hits, "Attach Add Input -> Datasets -> fatkhato/nascenia-code"
CODE = os.path.dirname(hits[0])

tr = glob.glob("/kaggle/input/**/train.parquet", recursive=True)
assert tr, "Attach Add Input -> Datasets -> fatkhato/nascenia-xfer-en"
DATA = os.path.dirname(tr[0])

os.makedirs("/kaggle/working/code", exist_ok=True)
for f in glob.glob(f"{CODE}/*.py"):
    shutil.copy(f, "/kaggle/working/code/")
sys.path.insert(0, "/kaggle/working/code")

import pandas as pd
for s in ("train", "dev", "test"):
    d = pd.read_parquet(f"{DATA}/{s}.parquet")
    print(f"  {s:5s} {len(d):7,d} {list(d.columns)}")
# The data is PREBUILT with the frozen seed-42 / 5,000-row split. 01_prep.py is NOT run here —
# rebuilding it would require the raw competition CSVs and risk a different split.
print("\nCODE:", CODE, "\nDATA:", DATA)

## 4 — Measure the real step time, then set the budget from it

mT5's 250k-token softmax makes step time hard to predict, and Kaggle hard-kills at 12 h — losing
everything. So: run a short timed slice at the **real** config, measure seconds/step, and derive how
many steps fit in **10.0 h**. Abort in minutes if even a useful run will not fit, rather than
discovering it at hour eleven.

In [ ]:
# ══ 4 — timed probe ═════════════════════════════════════════════════════════
import subprocess, shlex, os, time, json

RUN_ENV = {**os.environ, "CUDA_VISIBLE_DEVICES": "0",
           "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
           "TOKENIZERS_PARALLELISM": "false", "PYTHONUNBUFFERED": "1"}

PROBE_STEPS = 30
probe = (f"python 02_train_t5.py --data-dir {DATA} --out-dir /kaggle/working/probe "
         f"--model google/mt5-base --seed 11 "
         f"--max-train {PROBE_STEPS * 2 * 16} --epochs 1 "
         f"--lr 1e-3 --warmup 200 --optim adafactor "
         f"--batch-size 2 --grad-accum 16 --eval-batch-size 2 "
         f"--max-source-len 640 --max-target-len 448 "
         f"--eval-subset 40 --eval-steps 10000 --gen-num-beams 2 --gen-min-new-tokens 80 "
         f"--no-bertscore-eval --precision auto --no-group-by-length --run-name probe")
print(probe, flush=True)
t0 = time.time()
r = subprocess.run(shlex.split(probe), cwd="/kaggle/working/code", env=RUN_ENV)
elapsed = time.time() - t0
assert r.returncode == 0, "probe failed — do not start the full run"

SEC_PER_STEP = elapsed / PROBE_STEPS
print(f"\n{PROBE_STEPS} steps in {elapsed/60:.1f} min  ->  {SEC_PER_STEP:.1f} s/step")
print("   (includes model load + tokenisation, so this OVER-estimates the steady rate — good, it is the safe direction)")

In [ ]:
# ══ 5 — derive the step budget ══════════════════════════════════════════════
import pandas as pd
N_TRAIN = len(pd.read_parquet(f"{DATA}/train.parquet"))
EFF = 2 * 16
STEPS_PER_EPOCH = N_TRAIN / EFF

affordable = int(10.0 * 3600 / SEC_PER_STEP)
STEPS = min(affordable, 3000)          # 3000 is past where the incumbent peaked; no need for more
EPOCHS = round(STEPS / STEPS_PER_EPOCH, 3)

print(f"train rows       {N_TRAIN:,}   effective batch {EFF}   steps/epoch {STEPS_PER_EPOCH:,.0f}")
print(f"affordable in 10.0 h: {affordable:,} steps")
print(f"-> budget {STEPS:,} steps = {EPOCHS} epochs  (~{STEPS*SEC_PER_STEP/3600:.1f} h)")

# The incumbent needed ~2,750 optimizer updates at effective batch 64 to peak. Fewer updates than
# this cannot fairly test the hypothesis — a weak result would just mean undertrained.
assert STEPS >= 1200, (
    f"❌ only {STEPS} steps fit in 10.0 h at {SEC_PER_STEP:.1f} s/step. That is too "
    f"undertrained to interpret. Run this on a bigger GPU (see fine_tune_MT5_project/).")
if STEPS < 2000:
    print(f"\n⚠️  {STEPS} steps is below the ~2,750 updates the incumbent needed. A weak result "
          f"here may mean UNDERTRAINED rather than 'English does not help'.")

In [ ]:
# ══ 6 — FULL RUN ════════════════════════════════════════════════════════════
cmd = (f"python 02_train_t5.py --data-dir {DATA} --out-dir /kaggle/working/runs "
       f"--model google/mt5-base --seed 11 "
       f"--epochs {EPOCHS} --lr 1e-3 --warmup 200 --optim adafactor "
       f"--batch-size 2 --grad-accum 16 --eval-batch-size 2 "
       f"--max-source-len 640 --max-target-len 448 "
       f"--eval-subset 300 --eval-steps 250 "
       f"--gen-num-beams 4 --gen-min-new-tokens 80 "
       f"--precision auto --no-group-by-length --run-name mt5base_en_seed11")
print(cmd + "\n" + "=" * 70, flush=True)
t0 = time.time()
r = subprocess.run(shlex.split(cmd), cwd="/kaggle/working/code", env=RUN_ENV)
print(f"\nexit {r.returncode} after {(time.time()-t0)/60:.1f} min")
assert r.returncode == 0, "training failed — see traceback above"

In [ ]:
# ══ 7 — run record ══════════════════════════════════════════════════════════
import json, glob
for f in sorted(glob.glob("/kaggle/working/runs/*/run.json")):
    if "probe" in f:
        continue
    d = json.load(open(f))
    print(f"\n=== {f} ===")
    print(json.dumps(d, indent=2, ensure_ascii=False))
    dev = d.get("dev", {})
    if "token_f1" in dev:
        f1, rl = dev["token_f1"], dev["rouge_l"]
        print(f"\n  pred LB  {0.4646 + 0.3098*f1 + 0.2*rl:.4f}")
        print(f"  vs incumbent BanglaT5 draft-only: TokenF1 0.7724 / LB 0.85030  ->  {f1-0.7724:+.4f}")
        print(f"  vs no-model bar (draft + regexes): TokenF1 0.5984  ->  {f1-0.5984:+.4f}")

---
## After it finishes

Copy `run.json` and the best checkpoint's `trainer_state.json` into
`fine_tune_MT5_project/3_mt5base_english_draft/`, and fill in `RESULTS.md`.

**Read the result against the truncation caveat in cell 0.** If it lands below 0.7724, that is
consistent with *any* of: English does not help · mT5 is the wrong carrier · the 9% source
truncation · undertraining. Only the full-length run on a bigger GPU separates them.